### Step 1: Import Libraries and Load Environment Variables

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown #for displaying markdown in Jupyter Notebook

load_dotenv() #loading .env file into our virtual environment

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") #getting the value of this key from the .env file and storing it in the OPENAI_API_KEY variable (on LHS)

if OPENAI_API_KEY is None: #checking if the API Key is empty  
    raise Exception("API Key is missing.") #if it's empty, we'll get an error here.

#if this runs without errors, we are good. We can also add another line:
else: #if the key is there
    print(OPENAI_API_KEY[:8]) #shows the first [standard] 8 characters of the API Key

client = OpenAI()

sk-proj-


### Step 2: Call the API

In [3]:
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}, # Setting the System Prompt
        {"role": "user", "content": "Next week I'm travelling to Spain"} # User Prompt
    ]
)
reply = response.choices[0].message.content
display(Markdown(reply)) #displaying the response in markdown format in Jupyter Notebook

That sounds exciting! Do you need any tips or information for your trip to Spain? For example, recommendations on places to visit, local customs, language tips, or packing advice?

In [4]:
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}, #System Prompt
        {"role": "user", "content": "Where am I travelling to next week?"} #User Prompt
    ]
)
reply = response.choices[0].message.content #displaying the response in markdown format in Jupyter Notebook\
display(Markdown(reply))

I’m not sure where you are traveling to next week. Could you please provide more details?

It doesn't remember that you told it that you are travelling to Spain next week (**API call doesn't have memory**).
Every time you call the API (with the response and reply code above), you are calling it fresh; you are starting a new conversation with it every time you call it (by running the above code). A fresh API call every time.

Whereas LLM's (ChatGPT, Claude, etc), they know what is going on. Their code makes sure the AI is aware of what is happening or being discussed (it has memory).



### In Theory... (doing it manually for illustrative purposes)

In [5]:
messages = [ #list of messages that we will use for future interactions with the model
    {"role": "system", "content": "You are a helpful assistant."}, #storing this in the messages variable, so that we can reuse it in future interactions
    {"role": "user", "content": "Next week I'm travelling to Spain"}, #adding this line to the messages list, so that the model can remember this information for future interactions
    {"role": "assistant", "content":"That sounds exciting! Do you need any tips or information for your trip to Spain? For example, recommendations on places to visit, local customs, language help, or packing advice?"}, #adding the assistant's response to the messages list, so that the model can remember this information for future interactions
    {"role": "user", "content": "Where am I travelling to next week?"} #Adding the new prompt to the messages list for memory storage
] # assistant is the role of the model, and user is the role of the person interacting with the model
# system is the role of the instructions that we give to the model, which is to be a helpful assistant in this case.
# We put the reply the model gave for the user prompt of Next week I'm travelling to Spain in the Assistant Prompt line.

# The messages list is a way to keep track of the conversation history between the user and the assistant. Each message has a role (system, user, or assistant) and content (the actual message). By keeping track of the conversation history, we can have a more coherent conversation with the model, as it will remember the context of our previous interactions.
# By storing all of this information in the messages list, we can have a more coherent conversation with the model, as it will remember the context of our previous interactions.

The LLM isnt going to maintain the conversation history. We have to maintain it in our code and then pass it to the LLM.

In [6]:
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages #Our messages list
)

reply = response.choices[0].message.content #we're passing the last prompt of "Where am I travelling to next week?" to the model, and it should be able to remember the previous information about travelling to Spain, and give us the correct answer
# since we are passing the entire messages list (message history), the model has access to the entire conversation history, and can use that information to generate a more accurate response.
display(Markdown(reply))

You mentioned that you are travelling to Spain next week. Safe travels! If you need any help with your trip, feel free to ask.

This is peudo-memory (fake memory). We are spoon-feeding it the memory (explicitly writing it out and passing it in the code) and that's why it remembers.
The LLM is getting all the information again to its input (i.e., all of the model's response to "Next week I'm travelling to Spain" user prompt) and then saying You are travelling to Spain next week. 

**It's reprocessing your first message/prompt** ("Next week I'm travelling to Spain") **+ the response** ("That sounds exciting! Do you need any tips or information for your trip to Spain? For example, recommendations on places to visit, local customs, language help, or packing advice?") **+ the new prompt** ("Where am I travelling to next week?"), **and then its giving you the answer** ("You are travelling to Spain next week.").

In [12]:
messages = [ #list of messages that we will use for future interactions with the model
    {"role": "system", "content": "You are a helpful assistant."}, 
    {"role": "user", "content": "Next week I'm travelling to Spain"}, #adding this line to the messages list, so that the model can remember this information for future interactions
    {"role": "assistant", "content":"That sounds exciting! Do you need any tips or information for your trip to Spain? For example, recommendations on places to visit, local customs, language help, or packing advice?"}, #adding the assistant's response to the messages list, so that the model can remember what it said for future interactions
    {"role": "user", "content": "What is the capital?"}
] # assistant is the role of the model, and user is the role of the person interacting with the model

response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages
)

reply = response.choices[0].message.content #we're passing the last prompt of "What is the capital?" to the model, and it should be able to remember the previous information about travelling to Spain, and give us the correct answer
display(Markdown(reply))

The capital of Spain is Madrid. It's a vibrant city known for its rich history, art museums like the Prado, beautiful parks such as El Retiro, and lively nightlife. If you have time, it's definitely worth a visit!

### Now: NOT Manually

In [18]:
# Call the API
messages = [ # 2 messages:
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Next week I'm travelling to Spain."}
]

response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages #fine to have messages = messages bc the left one is the parameter name (passed into the chat.completions API), and the right one is the variable we created to store the list of messages
)

reply = response.choices[0].message.content

#Now take whatever the reply is and add it to the messages list, so that the model can remember this information for future interactions:
messages.append({"role": "assistant", "content": reply}) #adding the assistant's response to the messages list, so that the model can remember this information for future interactions
#append() is a method that adds an element to the end of a list. In this case, we are adding a new message to the messages list, with the role of "assistant" and the content of the reply that we got from the model.

# role is assistant bc it's the model's (or LLM's) response, and content is the actual message that the model generated in response to our prompt. By appending this new message to the messages list, we are keeping track of the conversation history, which allows us to have a more coherent conversation with the model in future interactions.

#display(Markdown(reply))
print(messages) #printing the messages list to see the conversation history, which now includes the user's initial message and the assistant's response. (Double checking)

# Start with messages only have 2 messages. Added another message to the messages list, which is the assistant's response to the user's initial message. Now we have 3 messages in the messages list, which allows us to have a more coherent conversation with the model in future interactions, as it will remember the context of our previous interactions.
# So there should be 3 messages in the messages list now: the system message, the user's initial message, and the assistant's response to that initial message.

[{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': "Next week I'm travelling to Spain."}, {'role': 'assistant', 'content': 'That sounds exciting! Do you need any tips or information for your trip to Spain? I can help with recommendations on places to visit, local customs, language tips, or anything else you might need.'}]


In [19]:
# To print this nicely:

from pprint import pprint #pprint is a module that allows us to print data structures in a more readable format. It stands for "pretty print". It is especially useful for printing nested data structures, such as lists and dictionaries, in a way that is easy to read and understand.
pprint(messages)

[{'content': 'You are a helpful assistant.', 'role': 'system'},
 {'content': "Next week I'm travelling to Spain.", 'role': 'user'},
 {'content': 'That sounds exciting! Do you need any tips or information for '
             'your trip to Spain? I can help with recommendations on places to '
             'visit, local customs, language tips, or anything else you might '
             'need.',
  'role': 'assistant'}]


In [20]:
#Alternatively:

for message in messages: #iterating through each message in the messages list
    role = message["role"] #getting the role of the message (system, user, or assistant)
    content = message["content"] #getting the content of the message (the actual message)
    print(f"{role.upper()}: {content}\n") #printing the role and content of each message in a nice format, with the role in uppercase for better visibility and new line after the content for better readability. This will give us a clear view of the conversation history, showing who said what in the conversation.

SYSTEM: You are a helpful assistant.

USER: Next week I'm travelling to Spain.

ASSISTANT: That sounds exciting! Do you need any tips or information for your trip to Spain? I can help with recommendations on places to visit, local customs, language tips, or anything else you might need.



In [21]:
# First, add the new user prompt to the current list of messages
messages.append({"role": "user", "content": "Where am I travelling to next week?"}) #adding a new user message to the messages list, with the role of "user" and the content of the new prompt that we want to ask the model. This allows us to keep track of the conversation history and have a more coherent conversation with the model in future interactions.

#pprint(messages) #printing the messages list to see the updated conversation history, which now includes the new user prompt that we just added.

# messages = messages[:3] #keeping only the first 4 messages in the messages list, which is useful if we want to limit the conversation history to a certain number of messages. This can help prevent the model from getting overwhelmed with too much information and can also help improve the performance of the model by keeping the conversation history manageable.
# prevents us from appending the same message multiple times if we run the same cell multiple times in a Jupyter Notebook, which can happen if we are experimenting with the code and want to see how the model responds to different prompts without having to worry about the conversation history getting too long or cluttered.

nunique = len(set([message["content"] for message in messages])) #getting the number of unique messages in the messages list, which can be useful for analyzing the conversation history and understanding how many different prompts or responses have been exchanged between the user and the assistant. This can help us identify patterns in the conversation and improve our interactions with the model.
print(nunique) #printing the number of unique messages in the messages list, which gives us an idea of how many different prompts or responses have been exchanged between the user and the assistant. This can help us understand the diversity of the conversation and identify areas where we may want to improve our interactions with the model.

messages = messages[:nunique] #keeping only the unique messages in the messages list, which can help us avoid repeating the same prompts or responses multiple times in the conversation history. This can help improve the performance of the model and make our interactions with it more efficient and effective.
pprint(messages)

4
[{'content': 'You are a helpful assistant.', 'role': 'system'},
 {'content': "Next week I'm travelling to Spain.", 'role': 'user'},
 {'content': 'That sounds exciting! Do you need any tips or information for '
             'your trip to Spain? I can help with recommendations on places to '
             'visit, local customs, language tips, or anything else you might '
             'need.',
  'role': 'assistant'},
 {'content': 'Where am I travelling to next week?', 'role': 'user'}]


In [ ]:
#Call the API
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages
)

reply = response.choices[0].message.content
messages.append({"role": "assistant", "content": reply}) #adding the assistant's response to the messages list, so that the model can remember this information for future interactions

nunique = len(set([(message["role"], message["content"]) for message in messages]))

messages = messages[:nunique] #keeping only the unique messages in the messages list (so that if we rerun the cell, we won't get duplicates in the messages list, improving effeciency of our code).

pprint(messages)
print(reply)

[{'content': 'You are a helpful assistant.', 'role': 'system'},
 {'content': "Next week I'm travelling to Spain.", 'role': 'user'},
 {'content': 'That sounds exciting! Do you need any tips or information for '
             'your trip to Spain? I can help with recommendations on places to '
             'visit, local customs, language tips, or anything else you might '
             'need.',
  'role': 'assistant'},
 {'content': 'Where am I travelling to next week?', 'role': 'user'},
 {'content': 'You are travelling to Spain next week.', 'role': 'assistant'},
 {'content': 'You mentioned that you are travelling to Spain next week. Safe '
             'travels! If you need any information or tips about Spain, feel '
             'free to ask.',
  'role': 'assistant'},
 {'content': 'You are travelling to Spain next week. Safe travels! If you need '
             'any recommendations or help preparing for your trip, just let me '
             'know.',
  'role': 'assistant'},
 {'content': 'You 

Note: LLM's are **non-deterministic** (exact output cannot be determine; it's variable. Changes each time). LLM's are stochastic by nature, meaning they use probabilities to predict the outcome and that results in a non-deterministic system. Therefore, everytime we run the code, we're going to get a slightly different answer.